<a href="https://colab.research.google.com/github/demsaid400-cpu/DI-BOOTCAMP/blob/main/Week7_Day2_Daily_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U pinecone==6.0.1 pinecone-notebooks

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 4.3 MB/s eta 0:00:00


In [ ]:
import os

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [ ]:
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")

pc = Pinecone(api_key=api_key)

In [ ]:
query = "Tell me about Apple's products"

documents = [
    "An apple is a nutritious fruit rich in fiber and vitamin C.",
    "Apple designs and sells products such as the iPhone, iPad, MacBook, Apple Watch, and AirPods.",
    "Apples come in many varieties including Fuji, Gala, and Granny Smith.",
    "Apple develops software like iOS, macOS, watchOS, and cloud services such as iCloud.",
    "Many people enjoy eating fresh apples as part of a healthy diet."
]

In [ ]:
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[
        {"id": str(i), "text": doc}
        for i, doc in enumerate(documents)
    ],
    top_n=3
)

In [ ]:
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")

    for i, m in enumerate(matches):
        print(f"{i+1}. Score: {m.score:.4f}")
        print(m.document.text)
        print()

show_reranked_results(query, reranked.data)

Query: Tell me about Apple's products

1. Score: 0.9547
Apple designs and sells products such as the iPhone, iPad, MacBook, Apple Watch, and AirPods.

2. Score: 0.5513
Apple develops software like iOS, macOS, watchOS, and cloud services such as iCloud.

3. Score: 0.0560
Apples come in many varieties including Fuji, Gala, and Granny Smith.



In [ ]:
!pip install pandas torch transformers

In [ ]:
import os
import time
import pandas as pd

from pinecone import Pinecone, ServerlessSpec

from transformers import AutoTokenizer, AutoModel

import torch

In [ ]:
cloud = os.getenv("PINECONE_CLOUD", "aws")
region = os.getenv("PINECONE_REGION", "us-east-1")

spec = ServerlessSpec(
    cloud=cloud,
    region=region
)

index_name = "medical-notes-index"

In [ ]:
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=spec
)

{
    "name": "medical-notes-index",
    "metric": "cosine",
    "host": "medical-notes-index-77scd04.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [ ]:
import requests
import tempfile

In [ ]:
with tempfile.TemporaryDirectory() as tmpdirname:

    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Using the correct URL for the Pinecone sample notes dataset
    url = "https://raw.githubusercontent.com/pinecone-io/examples/master/docs/data/sample_notes_data.jsonl"

    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(
        file_path,
        orient="records",
        lines=True
    )

In [ ]:
print("Data shape:", df.shape)

df.head()

Data shape: (100, 3)


,id,values,metadata
0,P011,"[-0.2027486265, 0.2769146562, -0.1509393603, 0...","{'advice': 'rest, hydrate', 'symptoms': 'heada..."
1,P001,"[0.1842793673, 0.4459365904, -0.0770567134, 0....","{'tests': 'EKG, stress test', 'symptoms': 'che..."
2,P002,"[-0.2040648609, -0.1739618927, -0.2897160649, ...","{'HbA1c': '7.2', 'condition': 'diabetes', 'med..."
3,P003,"[0.1889383644, 0.2924542725, -0.2335938066, -0...","{'symptoms': 'cough, wheezing', 'diagnosis': '..."
4,P004,"[-0.12171068040000001, 0.1674752235, -0.231888...","{'referral': 'dermatology', 'condition': 'susp..."


In [ ]:
index = pc.Index(name=index_name)

index.upsert_from_dataframe(df)

sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

{'upserted_count': 100}

In [ ]:
def is_fresh(index):

    stats = index.describe_index_stats()

    vector_count = stats.total_vector_count

    print("Vector count:", vector_count)

    return vector_count > 0


while not is_fresh(index):
    time.sleep(5)

print("Index ready!")

index.describe_index_stats()

Vector count: 100
Index ready!


{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}

In [ ]:
def get_embedding(input_question):

    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModel.from_pretrained(model_name)

    encoded_input = tokenizer(
        input_question,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        model_output = model(**encoded_input)

    embedding = model_output.last_hidden_state[0].mean(dim=0)

    return embedding

In [ ]:
question = "Patient has chest pain and shortness of breath."

query = get_embedding(question).tolist()

results = index.query(
    vector=[query],
    top_k=5,
    include_metadata=True
)

sorted_matches = sorted(
    results["matches"],
    key=lambda x: x["score"],
    reverse=True
)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
def show_results(question, matches):

    print(f"Question: '{question}'")

    print("\nResults:\n")

    for i, match in enumerate(matches):

        print(f"{i+1}. ID: {match['id']}")

        print(f"Score: {match['score']}")

        print(f"Metadata: {match['metadata']}")

        print()

show_results(question, sorted_matches)

Question: 'Patient has chest pain and shortness of breath.'

Results:

1. ID: P001
Score: 0.668714106
Metadata: {'symptoms': 'chest pain', 'tests': 'EKG, stress test'}

2. ID: P016
Score: 0.519489706
Metadata: {'condition': 'heart murmur', 'referral': 'cardiology'}

3. ID: P003
Score: 0.455608755
Metadata: {'diagnosis': 'bronchitis', 'symptoms': 'cough, wheezing', 'treatment': 'antibiotics'}

4. ID: P063
Score: 0.421102583
Metadata: {'diagnosis': 'pneumonia', 'symptoms': 'cough, fever', 'treatment': 'antibiotics'}

5. ID: P0100
Score: 0.415564418
Metadata: {'advice': 'over-the-counter pain relief, stretching', 'symptoms': 'muscle pain'}



In [ ]:
transformed_documents = [

    {
        "id": match["id"],
        "reranking_field": "; ".join(
            [
                f"{key}: {value}"
                for key, value in match["metadata"].items()
            ]
        )
    }

    for match in results["matches"]

]

In [ ]:
refined_query = "Patient requires treatment for severe chest pain."

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True
)

In [ ]:
def show_reranked_results(question, matches):

    print(f"Question: '{question}'")

    print("\nReranked Results:\n")

    for i, match in enumerate(matches):

        print(f"{i+1}. ID: {match.document.id}")

        print(f"Score: {match.score}")

        print(f"Reranking Field: {match.document.reranking_field}")

        print()

show_reranked_results(
    refined_query,
    reranked_results.data
)

Question: 'Patient requires treatment for severe chest pain.'

Reranked Results:

1. ID: P001
Score: 0.33906123
Reranking Field: symptoms: chest pain; tests: EKG, stress test

2. ID: P003
Score: 0.026455775
Reranking Field: diagnosis: bronchitis; symptoms: cough, wheezing; treatment: antibiotics

3. ID: P0100
Score: 0.010818449
Reranking Field: advice: over-the-counter pain relief, stretching; symptoms: muscle pain



In [ ]:
pc.delete_index(name=index_name)

In [ ]:
import nbformat

def clean_notebook_outputs(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Effacer les outputs de chaque cellule
    for cell in nb.cells:
        if cell.cell_type == 'code':
            cell.outputs = []
            cell.execution_count = None

    with open(output_file, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f"Fichier nettoyé créé : {output_file}")

# Note : Dans Colab, le fichier actuel n'est pas directement accessible par son nom
# sans sauvegarde préalable, mais vous pouvez utiliser cette méthode si vous uploadez le fichier
# ou simplement utiliser 'Modifier > Effacer tous les outputs' dans le menu en haut.